# Legal RAG Hackathon: Main Colab

Один основной ноутбук для setup, запуска скриптов и сохранения результатов в GitHub. Основная логика живет в `src/` и `scripts/`.

## 1. Setup

Задайте URL репозитория и путь, куда Colab будет его клонировать.

In [ ]:
from pathlib import Path

REPO_URL = "<PASTE_REPO_URL_HERE>"
REPO_DIR = Path("/content/legal-rag-hackathon")
COLAB_PATHS = "configs/paths.colab.yaml"

print(f"Repo dir: {REPO_DIR}")
print(f"Paths config: {COLAB_PATHS}")

## 2. Mount Google Drive

Данные читаются с Google Drive согласно `configs/paths.colab.yaml`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clone or Pull GitHub Repo

При первом запуске репозиторий клонируется, при повторном после рестарта выполняется `git pull`.

In [ ]:
%cd /content

if not REPO_DIR.exists():
    if REPO_URL == "<PASTE_REPO_URL_HERE>":
        raise ValueError("Set REPO_URL before cloning the repository.")
    repo_name = REPO_DIR.name
    !git clone $REPO_URL $repo_name

%cd $REPO_DIR
!git pull

## 4. Install Requirements

In [ ]:
%cd $REPO_DIR
!pip install -r requirements.txt

## 5. Check Data

In [ ]:
%cd $REPO_DIR
!python scripts/check_data.py --paths $COLAB_PATHS

## 6. Run Baseline

Скрипт сохранит train/test предсказания в `outputs/predictions/` и метрики в `outputs/metrics/`.

In [ ]:
%cd $REPO_DIR
!python scripts/run_baseline.py --paths $COLAB_PATHS

## 7. Make Submission

In [ ]:
%cd $REPO_DIR
!python scripts/make_submission.py --paths $COLAB_PATHS

## 8. Inspect Outputs

Этот блок удобно запускать после рестарта Colab, если нужно просто посмотреть уже сохраненные артефакты.

In [ ]:
%cd $REPO_DIR
!find outputs -maxdepth 2 -type f | sort

## 9. Save Outputs to GitHub

Если в `outputs/` или `experiments/` появились новые файлы, их можно закоммитить и запушить.

In [ ]:
%cd $REPO_DIR
!git status

import subprocess

tracked_paths = ["outputs", "experiments"]
status = subprocess.run(
    ["git", "status", "--porcelain", "--", *tracked_paths],
    capture_output=True,
    text=True,
    check=False,
)

if status.stdout.strip():
    subprocess.run(["git", "add", *tracked_paths], check=True)
    subprocess.run(["git", "commit", "-m", "add experiment outputs"], check=False)
    subprocess.run(["git", "push"], check=False)
else:
    print("Nothing new to commit in outputs/ or experiments/.")